# Salary Prediction - Demo Notebook

This notebook demonstrates how to use the salary prediction system.

## 1. Setup and Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

from salary_predictor import DataProcessor, SalaryPredictor, load_config

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Configuration

In [ ]:
config = load_config('../config.json')
print("Configuration loaded successfully!")
print(f"Model type: {config['model']['type']}")

## 3. Generate Sample Data

For demonstration purposes, we'll generate synthetic salary data.

In [ ]:
# Generate sample data
np.random.seed(42)
n_samples = 1000

data = {
    'years_of_experience': np.random.randint(0, 20, n_samples),
    'age': np.random.randint(22, 65, n_samples),
    'job_title': np.random.choice(['Software Engineer', 'Data Scientist', 'Product Manager', 'Designer'], n_samples),
    'company': np.random.choice(['IBM', 'Google', 'Microsoft', 'Amazon', 'Apple'], n_samples),
    'location': np.random.choice(['New York', 'San Francisco', 'Seattle', 'Austin', 'Boston'], n_samples),
    'education_level': np.random.choice(['Bachelor', 'Master', 'PhD'], n_samples)
}

# Generate salary based on features
base_salary = 50000
experience_factor = data['years_of_experience'] * 3000
education_bonus = [0 if x == 'Bachelor' else 10000 if x == 'Master' else 20000 for x in data['education_level']]
location_factor = [20000 if x in ['San Francisco', 'New York'] else 10000 for x in data['location']]
noise = np.random.normal(0, 5000, n_samples)

data['salary'] = base_salary + experience_factor + np.array(education_bonus) + np.array(location_factor) + noise

df = pd.DataFrame(data)
print(f"Generated {len(df)} samples")
df.head()

## 4. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(df.describe())

In [ ]:
# Salary distribution
plt.figure(figsize=(10, 6))
plt.hist(df['salary'], bins=50, edgecolor='black')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.title('Salary Distribution')
plt.show()

In [ ]:
# Salary vs Experience
plt.figure(figsize=(10, 6))
plt.scatter(df['years_of_experience'], df['salary'], alpha=0.5)
plt.xlabel('Years of Experience')
plt.ylabel('Salary')
plt.title('Salary vs Years of Experience')
plt.show()

## 5. Data Preprocessing

In [ ]:
# Initialize data processor
processor = DataProcessor()

# Prepare data
X_train, X_test, y_train, y_test = processor.prepare_data(
    df,
    target_column='salary',
    categorical_columns=['job_title', 'company', 'location', 'education_level'],
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 6. Model Training

In [ ]:
# Initialize and train model
model = SalaryPredictor(model_type='random_forest', n_estimators=100, max_depth=10, random_state=42)
model.train(X_train, y_train)

## 7. Model Evaluation

In [ ]:
# Evaluate model
metrics = model.evaluate(X_test, y_test)

In [ ]:
# Plot predictions vs actual
y_pred = model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs Predicted Salary')
plt.show()

## 8. Feature Importance

In [ ]:
# Get feature importance
feature_names = ['years_of_experience', 'age', 'job_title', 'company', 'location', 'education_level']
importance = model.get_feature_importance(feature_names)

if importance:
    features = list(importance.keys())
    scores = list(importance.values())
    
    plt.figure(figsize=(10, 6))
    plt.barh(features, scores)
    plt.xlabel('Importance Score')
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()

## 9. Make Predictions

In [ ]:
# Example prediction
sample_data = pd.DataFrame([{
    'years_of_experience': 5,
    'age': 28,
    'job_title': 'Software Engineer',
    'company': 'IBM',
    'location': 'New York',
    'education_level': 'Bachelor'
}])

# Process and predict
X_sample = processor.transform_new_data(sample_data, ['job_title', 'company', 'location', 'education_level'])
prediction = model.predict(X_sample)[0]

print(f"Predicted Salary: ${prediction:,.2f}")